In [ ]:
import ee
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    return collection.median().select(["B2", "B3", "B4", "B8"]).clip(mumbai)

# Fetch NO2 image
def fetch_no2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2") \
        .select('tropospheric_NO2_column_number_density') \
        .filterDate(start_date, end_date)
    return collection.mean().clip(mumbai)

# Sample training data
def sample_training_data(start_date, end_date, scale=100):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    no2_image = fetch_no2_image(start_date, end_date)

    combined_image = sentinel2_image.addBands(no2_image)
    samples = combined_image.sample(
        region=mumbai,
        scale=scale,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band in props for band in ["B2", "B3", "B4", "B8"]) and 'tropospheric_NO2_column_number_density' in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props['tropospheric_NO2_column_number_density'])

    return np.array(predictors), np.array(target)

# Train the Random Forest model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors)
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(predictors, target)
    return model, scaler

if __name__ == "__main__":
    start_date = '2018-06-28'
    end_date = datetime.now().strftime('%Y-%m-%d')

    print("Sampling training data...")
    predictors, target = sample_training_data(start_date, end_date, scale=100)

    print("Training the Random Forest model...")
    model, scaler = train_model(predictors, target)

    print("Saving model and scaler...")
    with open('/content/NO2_rf_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    np.save('/content/predictor_scaler_mean.npy', scaler.mean_)
    np.save('/content/predictor_scaler_scale.npy', scaler.scale_)

Sampling training data...
Training the Random Forest model...
Saving model and scaler...


In [ ]:
import ee
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Load trained Random Forest model and scalers
with open('/content/NO2_rf_model.pkl', 'rb') as f:
    model = pickle.load(f)
scaler_mean = np.load('/content/predictor_scaler_mean.npy')
scaler_scale = np.load('/content/predictor_scaler_scale.npy')

# Define Mumbai shapefile
mumbai_geometry = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file").geometry()

# Fetch image with fallbacks
def fetch_image(user_date):
    start_date = ee.Date(user_date)
    end_date = start_date.advance(7, 'day')

    # Sentinel-2
    s2_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(["B2", "B3", "B4", "B8"]).median()

    if s2_image.bandNames().size().getInfo() > 0:
        print("Using Sentinel-2 data.")
        return s2_image.clip(mumbai_geometry)

    # Landsat-8 fallback
    print("No Sentinel-2 data. Trying Landsat-8.")
    landsat_image = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']).median() \
        .multiply(0.0000275).add(-0.2).rename(['B2', 'B3', 'B4', 'B8'])

    if landsat_image.bandNames().size().getInfo() > 0:
        print("Using Landsat-8 data.")
        return landsat_image.clip(mumbai_geometry)

    # MODIS fallback
    print("No Landsat-8 data. Trying MODIS.")
    modis_image = ee.ImageCollection("MODIS/061/MCD43A4") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3', 'Nadir_Reflectance_Band2']) \
        .median().rename(['B2', 'B3', 'B4', 'B8'])

    if modis_image.bandNames().size().getInfo() > 0:
        print("Using MODIS data.")
        return modis_image.clip(mumbai_geometry)

    raise ValueError("No valid data available.")

# Predict and export NO₂
def predict_no2(user_date):
    print(f"Processing NO₂ for {user_date}")
    image = fetch_image(user_date)

    # Reduce the region to collect predictors
    predictors = image.sample(
        region=mumbai_geometry,
        scale=100,
        numPixels=5000,
        geometries=False
    ).getInfo()

    # Extract predictors and make predictions
    inputs = []
    for feature in predictors['features']:
        props = feature['properties']
        inputs.append([props['B2'], props['B3'], props['B4'], props['B8']])

    inputs = np.array(inputs)
    inputs = (inputs - scaler_mean) / scaler_scale
    predictions = model.predict(inputs).mean()

    # Create NO₂ image
    no2_image = ee.Image.constant(predictions).clip(mumbai_geometry).rename('Downscaled_NO2')

    print("Exporting downscaled NO₂...")
    task = ee.batch.Export.image.toAsset(
        image=no2_image,
        description='Downscaled_NO2_RF',
        assetId='projects/ee-chandanagalgali/assets/downscaled_NO2_rf',
        region=mumbai_geometry,
        scale=100,
        maxPixels=1e13
    )
    task.start()
    print("Export started! Monitor in GEE Tasks.")

if __name__ == "__main__":
    user_date = input("Enter the date for prediction (YYYY-MM-DD): ")
    predict_no2(user_date)
    print("Processing complete.")

Enter the date for prediction (YYYY-MM-DD): 2023-12-10
Processing NO₂ for 2023-12-10
Using Sentinel-2 data.
Exporting downscaled NO₂...
Export started! Monitor in GEE Tasks.
Processing complete.
